### Overview — `merge_era5_1979_2025.ipynb`

This notebook consolidates **processed ERA5 Excel files** into a single analysis-ready dataset covering 1979–2025:

- **Read & reshape per variable:** Loads each Excel file (with one sheet per year), converts from wide grids to **long format** (`latitude`, `longitude`, `datetime`), and concatenates across time blocks.  
- **Harmonize time:** Converts timestamps to proper datetimes, shifts from **UTC → Asia/Thimphu** (Bhutan local time), and enforces a **6-hourly cadence** (00, 06, 12, 18).  
- **Merge across variables:** Sequentially merges all variable tables on **(`latitude`, `longitude`, `datetime`)**, creating one aligned dataset with every variable.  
- **Write final artifact:** Exports to  
  `data/era5/era5_merged/merged_era5_6hour_1979_2025.parquet`  
  for fast access in modeling pipelines.

**Variables included:**  
`total_precipitation`, `runoff`, `surface_runoff`, `sub_surface_runoff`, `snowmelt`, `snow_depth`, `soil_temperature_level_1`, `surface_solar_radiation_downwards`, `2m_temperature` (°C), `2m_dewpoint_temperature`, `10m_u_component_of_wind`, `10m_v_component_of_wind`, `potential_evaporation`.


## 1. Imports

In [9]:

import pandas as pd
import os
from openpyxl import load_workbook
from tqdm import tqdm
from functools import reduce


## 2. Paths and Variable Mapping

In [10]:

# Path to your Excel folder
excel_dir = "../../data/era5/era5_data_excel"

# Mapping variable names to BOTH file paths (1979–1999 & 2000–2025)
variable_files = {
    "temperature": [
        "2m_temperature_6hour_1979_1999.xlsx",
        "2m_temperature_6hour_2000_2025.xlsx"
    ],
    "dewpoint": [
        "2m_dewpoint_temperature_6hour_1979_1999.xlsx",
        "2m_dewpoint_temperature_6hour_2000_2025.xlsx"
    ],
    "wind_u": [
        "10m_u_component_of_wind_6hour_1979_1999.xlsx",
        "10m_u_component_of_wind_6hour_2000_2025.xlsx"
    ],
    "wind_v": [
        "10m_v_component_of_wind_6hour_1979_1999.xlsx",
        "10m_v_component_of_wind_6hour_2000_2025.xlsx"
    ],
    "potential_evaporation": [
        "potential_evaporation_6hour_1979_1999.xlsx",
        "potential_evaporation_6hour_2000_2025.xlsx"
    ],
    "runoff": [
        "runoff_6hour_1979_1999.xlsx",
        "runoff_6hour_2000_2025.xlsx"
    ],
    "snow_depth": [
        "snow_depth_6hour_1979_1999.xlsx",
        "snow_depth_6hour_2000_2025.xlsx"
    ],
    "snowmelt": [
        "snowmelt_6hour_1979_1999.xlsx",
        "snowmelt_6hour_2000_2025.xlsx"
    ],
    "soil_temperature": [
        "soil_temperature_level_1_6hour_1979_1999.xlsx",
        "soil_temperature_level_1_6hour_2000_2025.xlsx"
    ],
    "sub_surface_runoff": [
        "sub_surface_runoff_6hour_1979_1999.xlsx",
        "sub_surface_runoff_6hour_2000_2025.xlsx"
    ],
    "surface_runoff": [
        "surface_runoff_6hour_1979_1999.xlsx",
        "surface_runoff_6hour_2000_2025.xlsx"
    ],
    "solar_radiation": [
        "surface_solar_radiation_downwards_6hour_1979_1999.xlsx",
        "surface_solar_radiation_downwards_6hour_2000_2025.xlsx"
    ],
    "precipitation": [
        "total_precipitation_6hour_1979_1999.xlsx",
        "total_precipitation_6hour_2000_2025.xlsx"
    ]
}


## 3. Function to Read and Melt

In [11]:

def read_and_melt_variable(filepath, var_name):
    all_years = []
    wb = load_workbook(filename=filepath, read_only=True)
    for sheet in tqdm(wb.sheetnames, desc=f"Loading {var_name}"):
        df = pd.read_excel(filepath, sheet_name=sheet)
        # reshape: latitude, longitude, datetime, value
        melted = df.melt(id_vars=["latitude", "longitude"],
                         var_name="datetime",
                         value_name=var_name)
        all_years.append(melted)
    result = pd.concat(all_years, ignore_index=True)
    # ensure datetime format
    result["datetime"] = pd.to_datetime(result["datetime"])
    return result


## 4. Load All Variables (1979–2025)

In [12]:

dfs = {}
for var, fnames in variable_files.items():
    parts = []
    for fname in fnames:
        full_path = os.path.join(excel_dir, fname)
        part = read_and_melt_variable(full_path, var)
        parts.append(part)
    dfs[var] = pd.concat(parts, ignore_index=True)
    print(var, dfs[var]["datetime"].min(), "→", dfs[var]["datetime"].max(), "rows:", len(dfs[var]))


Loading temperature: 100%|█████████████████████████████████████████████████████████████████████████████| 26/26 [00:22<00:00,  1.16it/s]


temperature 1979-01-01 00:00:00 → 2025-07-08 00:00:00 rows: 9174735


Loading dewpoint: 100%|████████████████████████████████████████████████████████████████████████████████| 26/26 [00:22<00:00,  1.17it/s]


dewpoint 1979-01-01 00:00:00 → 2025-07-08 00:00:00 rows: 9174735


Loading wind_u: 100%|██████████████████████████████████████████████████████████████████████████████████| 26/26 [00:21<00:00,  1.19it/s]


...

surface_runoff 1979-01-01 00:00:00 → 2025-07-08 00:00:00 rows: 9174735


Loading solar_radiation: 100%|█████████████████████████████████████████████████████████████████████████| 26/26 [00:20<00:00,  1.25it/s]


solar_radiation 1979-01-01 00:00:00 → 2025-07-07 18:00:00 rows: 9174600


Loading precipitation: 100%|███████████████████████████████████████████████████████████████████████████| 26/26 [00:21<00:00,  1.20it/s]


precipitation 1979-01-01 00:00:00 → 2025-07-07 18:00:00 rows: 9174600


## 5. Merge Across Variables

In [13]:

merged = reduce(
    lambda left, right: pd.merge(left, right, on=["latitude","longitude","datetime"], how="outer"),
    dfs.values()
)

merged.sort_values(["latitude","longitude","datetime"], inplace=True)
merged.reset_index(drop=True, inplace=True)

print(merged.shape)
merged.head()


(9183645, 16)


,latitude,longitude,datetime,temperature,dewpoint,wind_u,wind_v,potential_evaporation,runoff,snow_depth,snowmelt,soil_temperature,sub_surface_runoff,surface_runoff,solar_radiation,precipitation
0,26.5,88.5,1979-01-01 00:00:00,8.779205,280.687500,1.184982,-0.662811,1.918059e-06,0.000015,0.0,0.0,285.450195,0.000015,0.0,128.0,0.0
1,26.5,88.5,1979-01-01 06:00:00,23.847076,282.582336,-0.500717,-0.115112,-5.577810e-04,0.000016,0.0,0.0,294.535645,0.000016,0.0,2433472.0,0.0
2,26.5,88.5,1979-01-01 12:00:00,20.769440,282.469971,0.937103,-0.837021,-6.991206e-06,0.000016,0.0,0.0,293.682373,0.000016,0.0,0.0,0.0
3,26.5,88.5,1979-01-01 18:00:00,11.567841,281.223572,0.596100,-1.193222,-2.800487e-06,0.000016,0.0,0.0,287.762207,0.000016,0.0,0.0,0.0
4,26.5,88.5,1979-01-02 00:00:00,9.451782,281.199463,0.905502,-1.021103,3.932510e-07,0.000016,0.0,0.0,285.927246,0.000016,0.0,64.0,0.0


In [14]:
import pytz

# 🌏 Define Bhutan timezone
bhutan_tz = pytz.timezone('Asia/Thimphu')

# 🕒 Localize UTC and convert to Bhutan local time
merged['datetime'] = merged['datetime'].dt.tz_localize('UTC').dt.tz_convert(bhutan_tz)

## 6. Save to Parquet

In [16]:
import os

# Create folder if it doesn't exist
os.makedirs("../../data/era5/era5_merged", exist_ok=True)

# Save merged_df to Parquet
output_path = "../../data/era5/era5_merged/merged_era5_6hour_1979_2025.parquet"
merged.to_parquet(output_path)
